#Εξόρυξη Δεδομένων Άσκηση 3 Ερώτηση 3

ΟΜΑΔΑ: Αριστείδης Νικολακόπουλος (cs5308), Χρήστος Γιαμαλής (ma12834)

USING 2 FREE PASSES

Question3 Code:

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import networkx as nx
import warnings
from ast import literal_eval
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score, precision_score, recall_score, accuracy_score, make_scorer, confusion_matrix
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import cross_validate, train_test_split, KFold, cross_val_predict
from sklearn.pipeline import Pipeline
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import pairwise_distances
import gensim.downloader as api
from gensim.utils import simple_preprocess
from gensim.models import Word2Vec
warnings.filterwarnings("ignore")

RATINGS_PATH = 'ratings_small.csv'
LINKS_PATH = 'links_small.csv'
METADATA_PATH = 'movies_metadata.csv'


ratings = pd.read_csv(RATINGS_PATH)
links = pd.read_csv(LINKS_PATH)
metadata = pd.read_csv(METADATA_PATH, low_memory=False)

movie_counts = ratings['movieId'].value_counts()
valid_movies = movie_counts[movie_counts >= 10].index
ratings_filtered = ratings[ratings['movieId'].isin(valid_movies)]

def extract_genres(x):
    try:
        return [g['name'] for g in literal_eval(str(x))]
    except:
        return []

movie_user_matrix = ratings_filtered.pivot(index='movieId', columns='userId', values='rating').fillna(0)
print(f"Movie-User Matrix Shape: {movie_user_matrix.shape}")

print("\n Question 3: Classification")

target_genres = ['War', 'Music', 'Animation']

def get_single_genre(genre_list):
    matches = [g for g in genre_list if g in target_genres]
    if len(matches) == 1:
        return matches[0]
    return None

clf_df = metadata.copy()
clf_df['genres_list'] = clf_df['genres'].apply(extract_genres)
clf_df['target_genre'] = clf_df['genres_list'].apply(get_single_genre)
clf_df = clf_df.dropna(subset=['target_genre', 'overview'])

print(f"Classification Samples: {len(clf_df)}")
print(clf_df['target_genre'].value_counts())

X_text = clf_df['overview']
y = clf_df['target_genre']

classifiers = {
    "Decision Tree": DecisionTreeClassifier(),
    "kNN": KNeighborsClassifier(),
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "SVM": SVC(),
    "MLP": MLPClassifier(max_iter=500)
}

scoring_metrics = {
    'accuracy': 'accuracy',
    'precision': make_scorer(precision_score, average='weighted', zero_division=0),
    'recall': make_scorer(recall_score, average='weighted', zero_division=0)
}

print("\nTF-IDF Evaluation")
for name, clf in classifiers.items():
    pipeline = Pipeline([
        ('tfidf', TfidfVectorizer(stop_words='english', max_features=5000)),
        ('clf', clf)
    ])

    results = cross_validate(pipeline, X_text, y, cv=5, scoring=scoring_metrics)

    print(f"Model: {name}")
    print(f"Accuracy:  {results['test_accuracy'].mean():.4f}")
    print(f"Precision: {results['test_precision'].mean():.4f}")
    print(f"Recall:    {results['test_recall'].mean():.4f}")

    y_pred = cross_val_predict(pipeline, X_text, y, cv=5)
    cm = confusion_matrix(y, y_pred)
    print(f"Confusion Matrix:\n{cm}")

try:

    w2v_model = api.load("glove-wiki-gigaword-50")

    def get_doc_vector(doc):
        words = [w for w in simple_preprocess(doc) if w in w2v_model]

        if not words:
            return np.zeros(50)

        return np.mean(w2v_model[words], axis=0)

    X_w2v = np.vstack(X_text.apply(get_doc_vector))

    for name, clf in classifiers.items():
            results = cross_validate(clf, X_w2v, y, cv=5, scoring=scoring_metrics)
            print(f"Model: {name}")
            print(f"Accuracy:  {results['test_accuracy'].mean():.4f}")
            print(f"Precision: {results['test_precision'].mean():.4f}")
            print(f"Recall:    {results['test_recall'].mean():.4f}")

            y_pred = cross_val_predict(clf, X_w2v, y, cv=5)
            cm = confusion_matrix(y, y_pred)
            print(f"Confusion Matrix:\n{cm}")

except Exception as e:
    print(f"Skipping: {e}")

test_sentence = "A lonely soldier plays the piano while waiting for the battle to begin"
final_pipe = Pipeline([
    ('tfidf', TfidfVectorizer(stop_words='english', max_features=5000)),
    ('clf', LogisticRegression(max_iter=1000))
])
final_pipe.fit(X_text, y)
pred = final_pipe.predict([test_sentence])[0]
print(f"Prediction: {pred}")

Movie-User Matrix Shape: (2245, 671)

 Question 3: Classification
Classification Samples: 4630
target_genre
Animation    1837
Music        1510
War          1283
Name: count, dtype: int64

TF-IDF Evaluation
Model: Decision Tree
Accuracy:  0.7220
Precision: 0.7245
Recall:    0.7220
Confusion Matrix:
[[1214  382  241]
 [ 276 1110  124]
 [ 156   98 1029]]
Model: kNN
Accuracy:  0.7460
Precision: 0.7540
Recall:    0.7460
Confusion Matrix:
[[1419  250  168]
 [ 327 1046  137]
 [ 144  150  989]]
Model: Logistic Regression
Accuracy:  0.8683
Precision: 0.8718
Recall:    0.8683
Confusion Matrix:
[[1690  104   43]
 [ 235 1232   43]
 [ 122   63 1098]]
Model: SVM
Accuracy:  0.8624
Precision: 0.8683
Recall:    0.8624
Confusion Matrix:
[[1700   98   39]
 [ 251 1222   37]
 [ 153   59 1071]]
Model: MLP
Accuracy:  0.8443
Precision: 0.8445
Recall:    0.8443
Confusion Matrix:
[[1564  187   86]
 [ 208 1224   78]
 [  80   77 1126]]
Skipping: unable to read local cache '/Users/sharoukos_14/gensim-data/informa

#Results

TF-IDF Evaluation


Model: Decision Tree

Accuracy:  0.7190

Precision: 0.7211

Recall:    0.7190

Confusion Matrix:
[[1224  374  239]
 [ 283 1096  131]
 [ 163  110 1010]]

---
Model: kNN

Accuracy:  0.7460

Precision: 0.7540

Recall:    0.7460

Confusion Matrix:
[[1419  250  168]
 [ 327 1046  137]
 [ 144  150  989]]

---
Model: Logistic Regression

Accuracy:  0.8683

Precision: 0.8718

Recall:    0.8683

Confusion Matrix:
[[1690  104   43]
 [ 235 1232   43]
 [ 122   63 1098]]

---
Model: SVM

Accuracy:  0.8624

Precision: 0.8683

Recall:    0.8624

Confusion Matrix:
[[1700   98   39]
 [ 251 1222   37]
 [ 153   59 1071]]

---
Model: MLP

Accuracy:  0.8454

Precision: 0.8456

Recall:    0.8454

Confusion Matrix:
[[1568  187   82]
 [ 210 1221   79]
 [  80   83 1120]]

#Ανάλυση Αποτελεσμάτων

**Decision Tree**

Ο Decision Tree παρουσιάζει τη χαμηλότερη απόδοση. Αν και είναι εύκολα ερμηνεύσιμος, δυσκολεύεται να μοντελοποιήσει αποτελεσματικά δεδομένα υψηλής διάστασης όπως τα TF-IDF χαρακτηριστικά, με αποτέλεσμα αυξημένη σύγχυση μεταξύ των κατηγοριών.

**kNN**

Ο kNN επιτυγχάνει ελαφρώς καλύτερη απόδοση από το Decision Tree, ωστόσο επηρεάζεται αρνητικά από τη curse of dimensionality, καθώς η έννοια της απόστασης χάνει τη σημασία της σε χώρους υψηλής διάστασης.

**Logistic Regression**

Η Logistic Regression παρουσιάζει την καλύτερη συνολική απόδοση. Το μοντέλο εκμεταλλεύεται αποτελεσματικά τη γραμμική διαχωρισιμότητα που προσφέρει το TF-IDF representation και εμφανίζει υψηλή ακρίβεια και ισορροπία μεταξύ precision και recall. Τα confusion matrices δείχνουν ότι διαχωρίζει ικανοποιητικά και τις τρεις κατηγορίες.

**SVM**

Ο SVM παρουσιάζει παρόμοια απόδοση με τη Logistic Regression, ελαφρώς χαμηλότερη. Η καλή του συμπεριφορά είναι αναμενόμενη, καθώς οι SVMs είναι ιδιαίτερα αποδοτικοί σε προβλήματα κειμένου. Η μικρή διαφορά μπορεί να οφείλεται στις επιλεγμένες υπερπαραμέτρους.

**MLP**

Το MLP επιτυγχάνει καλή απόδοση, αλλά όχι ανώτερη από τα γραμμικά μοντέλα. Αυτό υποδηλώνει ότι η πληροφορία των περιλήψεων αποτυπώνεται επαρκώς σε γραμμικό χώρο και ότι η επιπλέον πολυπλοκότητα του νευρωνικού δικτύου δεν προσφέρει σημαντικό όφελος.